# Lot 41 — Deep-MCTS convergence & latency

Ce notebook lance le Lot 41 par étapes persistantes. Il ne fait aucun entraînement.

In [ ]:
from pathlib import Path
import os, subprocess, sys, time, json, select

REPO_URL = 'https://github.com/GlennEriss/songo-fish.git'
REPO_DIR = Path('/content/songo-fish')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[train,perf,dev]'], check=True)
os.environ['PYTHONPATH'] = 'packages:apps/trainer/scripts'
print('Projet installé depuis :', Path.cwd())

In [ ]:
from google.colab import drive, files
DRIVE_AVAILABLE = False
try:
    drive.mount('/content/drive')
    DRIVE_AVAILABLE = True
except Exception as exc:
    print('Drive indisponible, fallback téléchargement manuel:', exc)

DRIVE_ROOT = Path('/content/drive/MyDrive/songo-ai')
INPUT_BUNDLE = DRIVE_ROOT / 'inputs/lot41_inputs.tar.gz'
OUTPUT = DRIVE_ROOT / 'experiments/lot41_deep_mcts_convergence' if DRIVE_AVAILABLE else Path('/content/lot41_deep_mcts_convergence')
EXPORT = DRIVE_ROOT / 'exports/lot41_results.tar.gz' if DRIVE_AVAILABLE else Path('/content/lot41_results.tar.gz')
OUTPUT.mkdir(parents=True, exist_ok=True)
EXPORT.parent.mkdir(parents=True, exist_ok=True)
if INPUT_BUNDLE.exists():
    subprocess.run(['tar', '-xzf', str(INPUT_BUNDLE), '-C', str(REPO_DIR)], check=True)
print('OUTPUT =', OUTPUT)

In [ ]:
def supervised_run(cmd, label):
    log = Path('/content') / f'{label}.log'
    with log.open('wb') as stream:
        p = subprocess.Popen(cmd, stdout=stream, stderr=subprocess.STDOUT)
    off = 0
    started = time.time()
    while p.poll() is None:
        time.sleep(10)
        data = log.read_bytes()[off:]
        off += len(data)
        if data:
            print(data.decode(errors='replace'), end='', flush=True)
        print(f'[{label}] actif depuis {int(time.time()-started)}s', flush=True)
    data = log.read_bytes()[off:]
    if data:
        print(data.decode(errors='replace'), end='', flush=True)
    print(f'[{label}] code de sortie={p.returncode}; journal={log}', flush=True)
    if p.returncode:
        raise RuntimeError(f'{label} a échoué avec le code {p.returncode}; consulter {log}')

RUN = [sys.executable, 'apps/trainer/scripts/run_srn_lot41.py']

In [ ]:
supervised_run(RUN + ['--stage','prepare','--device','cuda','--positions-file','data/colab_bridge/lot39_benchmark_positions.json','--output',str(OUTPUT)], 'lot41_prepare')
supervised_run(RUN + ['--stage','pilot','--device','cuda','--output',str(OUTPUT)], 'lot41_pilot')

In [ ]:
supervised_run(RUN + ['--stage','stage_a','--device','cuda','--output',str(OUTPUT)], 'lot41_stage_a')

In [ ]:
supervised_run(RUN + ['--stage','stage_b','--device','cuda','--output',str(OUTPUT)], 'lot41_8192')

In [ ]:
supervised_run(RUN + ['--stage','stage_c','--device','cuda','--output',str(OUTPUT)], 'lot41_16384')

In [ ]:
supervised_run(RUN + ['--stage','stage_d','--device','cuda','--output',str(OUTPUT)], 'lot41_32768')

In [ ]:
supervised_run(RUN + ['--stage','stage_e','--device','cuda','--output',str(OUTPUT)], 'lot41_65536')

In [ ]:
supervised_run(RUN + ['--stage','latency','--device','cuda','--output',str(OUTPUT)], 'lot41_latency')

In [ ]:
supervised_run(RUN + ['--stage','finalize','--output',str(OUTPUT)], 'lot41_finalize')
supervised_run(RUN + ['--stage','export','--output',str(OUTPUT),'--bundle',str(EXPORT)], 'lot41_export')
if not DRIVE_AVAILABLE:
    files.download(str(EXPORT))
    files.download(str(Path(str(EXPORT) + '.sha256'))